Reference: https://www.10xgenomics.com/analysis-guides/xenium-downstream-analysis-in-python-tutorial

This notebook continues from `01_Xenium_5k_data_analysis_journey_python.ipynb` (Section 2.4): downstream analysis of the count matrix (QC, normalization, clustering, marker genes).

In [ ]:
# Import the python packages. Rerun this code again if you encounter an error on xarray_schema.
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import spatialdata as sd
import spatialdata_plot

%load_ext autoreload
%autoreload 2

warnings.filterwarnings("ignore")

In [ ]:
sdata = sd.SpatialData.read("../../data/spatialdata/xenium_prime_5k.zarr")

### 2.4 Downstream analysis of count matrix (sdata['table'])

The first step is QC to remove cells with low transcript counts. We plot the distribution of transcript counts per cell to pick a cutoff (subjective, no gold-standard rule).

We shallow-copy with `adata = sdata['table']` — changes to `adata` are reflected in `sdata['table']`. To keep a separate copy, use `adata = sdata['table'].copy()`.

In [ ]:
adata = sdata["table"]
# We keep the raw counts in a layer for later use
adata.layers["counts"] = adata.X.copy()

In [ ]:
# Visualize the per-cell transcript count distribution to choose a QC cutoff
fig = plt.hist(adata.obs["total_counts"], range=(0, 200), bins=100)
plt.axvline(x=20, color="r", linestyle="--")  # Adjust the cutoff based on the histogram

plt.xlabel("Transcripts per cell")
plt.ylabel("Number of cells")

In [ ]:
# Based on the histogram above, we use 20 transcripts as the lower cutoff
thres = np.quantile(adata.obs["total_counts"], 0.98)
sc.pp.filter_cells(adata, min_counts=20)
sc.pp.filter_cells(adata, max_counts=thres)

# We also filter out genes that are rarely expressed
sc.pp.filter_genes(adata, min_cells=100)

In [ ]:
%%time
# This is a Xenium Prime panel (>5k genes), so we can find the top 2k highly variable genes
sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=2000)

In [ ]:
%%time
# Log Normalization
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.layers["lognorm"] = adata.X.copy()

In [ ]:
%%time
# We don't center the data here since PCA handles centering, keeping X sparse for memory efficiency
sc.pp.scale(adata, zero_center=False, max_value=10)

In [ ]:
%%time
# PCA for dimension reduction, followed by UMAP/neighbors for downstream analysis
sc.pp.pca(adata, n_comps=30)
sc.pp.neighbors(adata, metric="cosine")
# igraph flavor scales better on larger Xenium datasets. Resolution controls the number of clusters
sc.tl.leiden(adata, flavor="igraph", n_iterations=-1, resolution=0.5)

In [ ]:
%%time
# Can take a while on the full sample
sc.tl.umap(adata)

In [ ]:
# Visualize the Leiden clusters on the UMAP embedding
sc.pl.umap(adata, color="leiden")

#### Plot clustering results spatially in the slide

In [ ]:
# Re-link the table to cell_boundaries so Leiden clusters can be rendered spatially
sdata.tables["table"].obs["region"] = "cell_boundaries"
sdata.set_table_annotates_spatialelement("table", region="cell_boundaries")

sdata.pl.render_shapes("cell_boundaries", color="leiden").pl.show(figsize=(12.8, 9.6))

#### Export clustering results (for napari / Xenium Explorer)

Save clustering results in a CSV file with two columns — `cell_id` and `group`.

In [ ]:
# Export cell_id/cluster pairs so results can be loaded back into napari or Xenium Explorer
clustering_res = adata.obs[["cell_id", "leiden"]]
clustering_res.columns = ["cell_id", "group"]
clustering_res.to_csv("../../data/xenium_prime_5k_clustering.csv", index=False)
clustering_res.head()

#### Find marker genes for each cluster

`sc.tl.rank_genes_groups` finds marker genes per cluster. `pts=True` also returns the percentage of cells expressing each gene. It expects log-normalized data, so we use `layer="lognorm"`.

In [ ]:
# Compute differentially expressed marker genes for each Leiden cluster
sc.tl.rank_genes_groups(adata, groupby="leiden", layer="lognorm", pts=True)

In [ ]:
# Dotplot of the top 5 marker genes per cluster
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5)

#### Cell annotation based on marker genes in each cluster

Based on the marker genes above, annotate each cluster:

1. Get the marker genes for a cluster.
2. Find associated cell types via a third-party tool, e.g. [Enrichr](https://maayanlab.cloud/Enrichr/).
3. Confirm cell identity using spatial location in napari (together with the H&E image if available).

Below is a template for pulling the full marker gene table for a single cluster — replace `"0"` with the cluster you're investigating.

In [ ]:
# Pull the full ranked marker gene table for a single cluster (edit group id as needed)
cluster_markers = sc.get.rank_genes_groups_df(adata, group="0")
cluster_markers.head(10)

#### Saving the processed data

We save the processed Zarr store for further use in Section 3.

In [ ]:
# Save the processed SpatialData (with QC, clustering, etc.) for reuse in Section 3
sdata.write("../../data/xenium_prime_5k_processed.zarr", overwrite=True)

## 3. Additional topics: neighborhood enrichment & spatially variable genes

Here we reuse the single lung sample processed in Section 2 to demonstrate a simple neighborhood enrichment and spatial autocorrelation analysis.

In [ ]:
# Read the processed sample saved at the end of Section 2
sdata = sd.read_zarr("../../data/xenium_prime_5k_processed.zarr")
adata = sdata["table"]

### 3.1 Centrality scores

Centrality measures how "important" a cluster is within the spatial neighborhood graph.

- **Average clustering** — tendency of cluster members to form triangles; high values suggest tight-knit, localized tissue regions.
- **Closeness centrality** — how close a cluster is, on average, to all other clusters; a high value may indicate a mediator role in tissue structure.
- **Degree centrality** — fraction of edges from cluster members that connect to other clusters; high values suggest outward-facing influence.

In [ ]:
# Build the spatial neighbor graph (Delaunay triangulation), then compute per-cluster centrality metrics
sq.gr.spatial_neighbors(adata, coord_type="generic", delaunay=True)
sq.gr.centrality_scores(adata, cluster_key="leiden")
sq.pl.centrality_scores(adata, cluster_key="leiden", figsize=(16, 5))

### 3.2 Neighborhood enrichment

Neighborhood enrichment scores the proximity of cell clusters on the connectivity graph against a permutation-based null distribution, producing a z-score per cluster pair.

In [ ]:
# Test whether cluster pairs are spatially enriched (permutation-based z-scores)
sq.gr.nhood_enrichment(adata, cluster_key="leiden")

fig, ax = plt.subplots(1, 2, figsize=(13, 7))
sq.pl.nhood_enrichment(
    adata,
    cluster_key="leiden",
    figsize=(8, 8),
    title="Neighborhood enrichment adata",
    ax=ax[0],
)
sdata.pl.render_shapes("cell_boundaries", color="leiden").pl.show(ax=ax[1])

### 3.3 Identify spatially variable genes (Moran's I)

A high Moran's I score indicates strong, non-random, spatially clustered gene expression — a useful signal for tissue organization and spatial domains.

In [ ]:
%%time
# Compute Moran's I spatial autocorrelation per gene to find spatially variable genes
sq.gr.spatial_autocorr(adata, mode="moran", n_jobs=-1)
adata.uns["moranI"].head(10)

In [ ]:
# Auto-pick the top 2 spatially variable genes instead of hardcoding gene names
top_genes = adata.uns["moranI"].index[:2].tolist()

sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    color=top_genes,
    shape=None,
    size=2,
    img=False,
    layer="lognorm",
)